In [ ]:
%%bash
  # Cell 1: Mount Drive
  from google.colab import drive
  drive.mount('/content/drive')
  !mkdir -p "/content/drive/MyDrive/p53_md_backup"

  # Cell 2: Run simulation with auto-backup (paste as one cell)
  %%bash
  cd /content/md_simulation

  # Start simulation in background
  gmx mdrun -v -deffnm md -cpi md.cpt -ntomp 4 -nb gpu -nstlist 300 -noappend &
  MDRUN_PID=$!

  # Backup loop every 5 minutes
  while kill -0 $MDRUN_PID 2>/dev/null; do
      sleep 300
      echo "=== Backing up to Google Drive ==="
      cp -v md.cpt md.part*.xtc md.part*.edr md.part*.log md.tpr /content/drive/MyDrive/p53_md_backup/ 2>/dev/null
      echo "=== Backup complete ==="
  done

  # Final backup when done
  echo "=== Simulation finished! Final backup ==="
  cp -v md.cpt md.part*.xtc md.part*.edr md.part*.log md.tpr /content/drive/MyDrive/p53_md_backup/
  echo "=== All done! ==="

bash: line 2: from: command not found
bash: line 3: syntax error near unexpected token `'/content/drive''
bash: line 3: `  drive.mount('/content/drive')                                                                                                                                                      '


CalledProcessError: Command 'b'  # Cell 1: Mount Drive                                                                                                                                                              \n  from google.colab import drive                                                                                                                                                     \n  drive.mount(\'/content/drive\')                                                                                                                                                      \n  !mkdir -p "/content/drive/MyDrive/p53_md_backup"                                                                                                                                   \n                                                                                                                                                                                     \n  # Cell 2: Run simulation with auto-backup (paste a

In [ ]:
!


drive  md.cpt  md.edr  md.log  md.tpr  md.xtc  sample_data  topol.top


In [ ]:
  # Cell 1: Mount Drive
  from google.colab import drive
  drive.mount('/content/drive')
  !mkdir -p "/content/drive/MyDrive/p53_md_backup"

Mounted at /content/drive


# GROMACS MD Simulation on Google Colab (Free T4 GPU)

**Goal:** Run 10 ns MD simulation using free T4 GPU  
**Speedup:** 5-10× faster than Mac M1 Pro CPU  
**Time:** ~2-3 hours total (including setup)  
**Cost:** FREE!

---

## ⚠️ CRITICAL: Enable GPU First!

1. Click **Runtime** → **Change runtime type**
2. **Hardware accelerator** → **GPU** (T4)
3. Click **Save**

---

## Why Compile from Source?

**Important:** Pre-built conda packages do NOT include GPU support!  
([Source: GROMACS Forums](https://gromacs.bioexcel.eu/t/gromacs-install-with-conda-but-cant-run-on-gpu/10856))

We must compile GROMACS with `-DGMX_GPU=CUDA` flag for GPU acceleration.  
This takes ~20-30 minutes but is the **only reliable method**.

---

## Timeline

| Step | Time | Description |
|------|------|-------------|
| 1. Verify GPU | 10 sec | Check T4 is available |
| 2. Install dependencies | 2-3 min | cmake, build tools |
| 3. Compile GROMACS | 20-30 min | Build with CUDA |
| 4. Mount Drive | 30 sec | Access your files |
| 5. Upload/verify files | 1-2 min | Get simulation files |
| 6. Run simulation | 1-2 hours | The actual MD |
| 7. Save results | 5 min | Copy to Drive |

**Total: ~2-3 hours** (vs 12+ hours on CPU!)

---


## Step 1: Verify GPU is Available

**Time:** 10 seconds

Must see "Tesla T4" or similar NVIDIA GPU.

In [ ]:
# Check NVIDIA GPU
!nvidia-smi

print("\n" + "="*70)

import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,compute_cap',
                        '--format=csv,noheader'], capture_output=True, text=True)

if result.returncode == 0:
    gpu_info = result.stdout.strip()
    print(f"\n✅ GPU Detected: {gpu_info}")

    # Check CUDA version
    cuda_result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
    if cuda_result.returncode == 0:
        for line in cuda_result.stdout.split('\n'):
            if 'release' in line:
                print(f"✅ CUDA: {line.strip()}")

    print("\n🎉 GPU ready! Proceed to Step 2.")
else:
    print("\n❌ ERROR: No GPU detected!")
    print("\n⚠️  FIX THIS NOW:")
    print("   1. Runtime → Change runtime type")
    print("   2. Hardware accelerator → GPU")
    print("   3. Save → Re-run this cell")

print("="*70)

Thu Jan 29 04:15:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Step 2: Install Build Dependencies

**Time:** 2-3 minutes

Install cmake, compilers, and libraries needed to build GROMACS.

In [1]:
%%time
print("📦 Installing build dependencies...\n")

# Update package list
!apt-get update -qq

# Install required packages
!apt-get install -y -qq \
    cmake \
    build-essential \
    libfftw3-dev \
    libopenmpi-dev \
    openmpi-bin \
    > /dev/null 2>&1

# Verify cmake version
print("\n📋 Checking installed tools:")
!cmake --version | head -1
!gcc --version | head -1
!nvcc --version | grep release

print("\n✅ Dependencies installed!")

📦 Installing build dependencies...

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

📋 Checking installed tools:
cmake version 3.31.10
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0
Cuda compilation tools, release 12.5, V12.5.82

✅ Dependencies installed!
CPU times: user 36.6 ms, sys: 6.19 ms, total: 42.8 ms
Wall time: 16.1 s


## Step 3: Download and Compile GROMACS with CUDA

**Time:** 20-30 minutes

This is the critical step that enables GPU acceleration.  
We compile GROMACS 2024.4 with `-DGMX_GPU=CUDA` flag.

☕ **Go get a coffee** - this takes a while but only needs to run once per session.

In [ ]:
%%bash
  # Download and extract GROMACS
  wget https://ftp.gromacs.org/gromacs/gromacs-2025.2.tar.gz
  tar xfz gromacs-2025.2.tar.gz
  cd gromacs-2025.2
  mkdir build
  cd build

  # Configure for A100 (compute capability 8.0)
  cmake .. \
           -DGMX_BUILD_OWN_FFTW=ON \
           -DGMX_GPU=CUDA \
           -DCUDA_TOOLKIT_ROOT_DIR=/usr/local/cuda \
           -DGMX_CUDA_TARGET_SM="80" \
           -DCMAKE_INSTALL_PREFIX=/usr/local/gromacs

  # Compile (use all available cores)
  make -j$(nproc)
  make install

  # Source the GROMACS environment (this only applies to this bash session)
  source /usr/local/gromacs/bin/GMXRC

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Python3: /usr/local/bin/python (found suitable version "3.12.12", minimum required is "3.9") found components: Interpreter Development Development.Module Development.Embed
-- Selected GPU FFT library - cuFFT
-- Found OpenMP_C: -fopenmp (found version "4.5")
-- Found OpenMP_CXX: -fopenmp (found version "4.5")
-- Found OpenMP: TRUE (found version "4.5")
-- Performing Test CFLAGS_WARN_NO_MISSING_FIELD_INITIALIZERS
-- Performing Test CFLAGS_WARN_NO_MISSING_FIELD_INITIALIZERS - Success


--2026-01-29 19:30:16--  https://ftp.gromacs.org/gromacs/gromacs-2025.2.tar.gz
Resolving ftp.gromacs.org (ftp.gromacs.org)... 130.237.11.165, 2001:6b0:1:1191:216:3eff:fec7:6e30
Connecting to ftp.gromacs.org (ftp.gromacs.org)|130.237.11.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 44447261 (42M) [application/x-gzip]
Saving to: ‘gromacs-2025.2.tar.gz’

     0K .......... .......... .......... .......... ..........  0%  235K 3m5s
    50K .......... .......... .......... .......... ..........  0%  528K 2m13s
   100K .......... .......... .......... .......... ..........  0%  645K 1m51s
   150K .......... .......... .......... .......... ..........  0% 1.19M 92s
   200K .......... .......... .......... .......... ..........  0% 1.20M 81s
   250K .......... .......... .......... .......... ..........  0% 1.61M 71s
   300K .......... .......... .......... .......... ..........  0% 1.91M 64s
   350K .......... .......... .......... .......... ..........  0% 1.9

In [ ]:
  !ps aux | grep make | grep -v grep || echo "Compilation finished or not started"
  !ls -la /content/gromacs_build/gromacs-2024.4/build/bin/gmx 2>/dev/null && echo "✅ gmx binary exists!" || echo "⏳ Still compiling..."

Compilation finished or not started
⏳ Still compiling...


## Step 3b: Verify GPU Support is Enabled

**Time:** 10 seconds

**CRITICAL:** Verify GROMACS was compiled with CUDA support.

In [2]:
import os
import subprocess

# Make sure PATH is set
os.environ['PATH'] = f"/usr/local/gromacs/bin:{os.environ['PATH']}"

print("📋 GROMACS Version and GPU Support:\n")

# Get version info
result = subprocess.run(['/usr/local/gromacs/bin/gmx', '--version'],
                       capture_output=True, text=True)
output = result.stdout

# Print key lines
important_keys = ['GROMACS version', 'GPU support', 'CUDA', 'OpenCL', 'SIMD']
for line in output.split('\n'):
    for key in important_keys:
        if key in line:
            print(f"  {line.strip()}")
            break

print("\n" + "="*70)

# Check for CUDA support
if 'CUDA' in output and ('enabled' in output.lower() or 'GPU support' in output):
    print("✅ SUCCESS: GROMACS has CUDA GPU support enabled!")
    print("   Ready to run simulations with GPU acceleration.")
else:
    print("⚠️  WARNING: CUDA support may not be enabled.")
    print("   Check full output above.")
    print("\nFull version output:")
    print(output)

print("="*70)

📋 GROMACS Version and GPU Support:

  GROMACS version:     2025.2
  GPU support:         CUDA
  SIMD instructions:   AVX_512
  CUDA compiler:       /usr/local/cuda/bin/nvcc nvcc: NVIDIA (R) Cuda compiler driver;Copyright (c) 2005-2024 NVIDIA Corporation;Built on Thu_Jun__6_02:18:23_PDT_2024;Cuda compilation tools, release 12.5, V12.5.82;Build cuda_12.5.r12.5/compiler.34385749_0
  CUDA compiler flags: -O3 -DNDEBUG
  CUDA driver:         12.40
  CUDA runtime:        12.50

✅ SUCCESS: GROMACS has CUDA GPU support enabled!
   Ready to run simulations with GPU acceleration.


## Step 4: Mount Google Drive

**Time:** 30 seconds

Click the authorization link when prompted.

In [3]:
from google.colab import drive

print("📁 Mounting Google Drive...\n")
drive.mount('/content/drive')

print("\n✅ Google Drive mounted!")
print("   Your files are at: /content/drive/MyDrive/")

📁 Mounting Google Drive...

Mounted at /content/drive

✅ Google Drive mounted!
   Your files are at: /content/drive/MyDrive/


## Step 5: Upload Simulation Files

**Time:** 1-2 minutes

### Option A: Upload directly to Colab (EASIEST)
1. Click the **folder icon** in left sidebar
2. Click **upload icon** (arrow pointing up)
3. Upload these files:
   - `md.tpr` (9.5 MB)
   - `md.cpt` (8.5 MB) - checkpoint file
   - `topol.top` (873 KB)

### Option B: Upload to Google Drive first
1. Go to https://drive.google.com
2. Create folder: `p53_simulation`
3. Upload files there
4. Run cell below to copy them

In [5]:
import os

# Create working directory
work_dir = '/content/md_simulation'
!mkdir -p {work_dir}

print("📁 Setting up simulation files...\n")

# Check Option A: Files uploaded directly to Colab
colab_files = ['/content/md.tpr', '/content/md.cpt', '/content/topol.top']
colab_found = all(os.path.exists(f) for f in colab_files)

# Check Option B: Files in Google Drive
drive_dir = '/content/drive/MyDrive/p53_simulation'
drive_files = [f'{drive_dir}/md.tpr', f'{drive_dir}/md.cpt', f'{drive_dir}/topol.top']
drive_found = all(os.path.exists(f) for f in drive_files)

if colab_found:
    print("✅ Found files uploaded to Colab")
    !cp /content/md.tpr /content/md.cpt /content/topol.top {work_dir}/
    print("   Copied to working directory.")
elif drive_found:
    print("✅ Found files in Google Drive")
    !cp {drive_dir}/* {work_dir}/
    print("   Copied to working directory.")
else:
    print("❌ Files not found!\n")
    print("📥 Please upload your files using ONE of these methods:\n")
    print("   Option A (Easiest): Upload directly to Colab")
    print("   - Click folder icon (📁) in left sidebar")
    print("   - Click upload icon (⬆️)")
    print("   - Select: md.tpr, md.cpt, topol.top\n")
    print("   Option B: Upload to Google Drive")
    print(f"   - Create folder: {drive_dir}")
    print("   - Upload files there\n")
    print("   Then re-run this cell.")

# Verify files
print("\n" + "="*70)
print("📋 Files in working directory:")
!ls -lh {work_dir}/

# Check file sizes
required_files = {
    'md.tpr': (9.0, 10.0),
    'md.cpt': (8.0, 9.0),
    'topol.top': (0.8, 1.0)
}

all_ok = True
for filename, (min_mb, max_mb) in required_files.items():
    filepath = f'{work_dir}/{filename}'
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / 1024 / 1024
        if size_mb < 0.1:  # File too small (probably LFS pointer)
            print(f"⚠️  {filename} is too small ({size_mb:.3f} MB) - may be corrupted!")
            all_ok = False
    else:
        all_ok = False

if all_ok:
    print("\n✅ All files ready! Proceed to Step 6.")
print("="*70)

📁 Setting up simulation files...

❌ Files not found!

📥 Please upload your files using ONE of these methods:

   Option A (Easiest): Upload directly to Colab
   - Click folder icon (📁) in left sidebar
   - Click upload icon (⬆️)
   - Select: md.tpr, md.cpt, topol.top

   Option B: Upload to Google Drive
   - Create folder: /content/drive/MyDrive/p53_simulation
   - Upload files there

   Then re-run this cell.

📋 Files in working directory:
total 54M
-rw-r--r-- 1 root root 8.5M Jan 30 04:55  md.cpt
-rw-r--r-- 1 root root 117K Jan 30 04:55  md.part0007.edr
-rw-r--r-- 1 root root 126K Jan 30 04:55  md.part0007.log
-rw-r--r-- 1 root root 1.9K Jan 30 04:55  md.part0008.edr
-rw-r--r-- 1 root root  25K Jan 30 04:55  md.part0008.log
-rw-r--r-- 1 root root  67K Jan 30 04:55  md.part0009.edr
-rw-r--r-- 1 root root  82K Jan 30 04:55  md.part0009.log
-rw-r--r-- 1 root root 138K Jan 30 04:55  md.part0010.edr
-rw-r--r-- 1 root root 143K Jan 30 04:55  md.part0010.log
-rw-r--r-- 1 root root 3.1K Jan 

## Step 6: Run MD Simulation with GPU 🚀

**Time:** 1-2 hours for 10 ns (or remaining from checkpoint)

**⚠️ IMPORTANT:**
- This runs for 1-2 hours
- Progress updates every 20 ps
- Don't close browser tab!
- Colab free tier: max 12-hour sessions

### Expected Performance
- **T4 GPU:** 150-250 ns/day
- **Mac M1 CPU:** 26 ns/day
- **Speedup:** 6-10×!

In [8]:
  import subprocess
  import time
  import shutil
  import os
  from datetime import datetime

  # Ensure PATH is set with GROMACS binaries
  os.environ['PATH'] = f"/usr/local/gromacs/bin:{os.environ['PATH']}"

  # Paths
  WORK_DIR = "/content/md_simulation"
  BACKUP_DIR = "/content/drive/MyDrive/p53_md_backup"

  # Files to backup
  BACKUP_FILES = ["md.cpt", "md.part*.xtc", "md.part*.edr", "md.part*.log", "md.tpr"]

  def backup_to_drive():
      """Copy checkpoint and output files to Google Drive"""
      timestamp = datetime.now().strftime("%H:%M:%S")
      print(f"[{timestamp}] Backing up to Google Drive...")

      for pattern in BACKUP_FILES:
          if "*" in pattern:
              # Handle wildcard patterns
              import glob
              files = glob.glob(f"{WORK_DIR}/{pattern}")
              for f in files:
                  shutil.copy2(f, BACKUP_DIR)
                  print(f"  Copied {os.path.basename(f)}")
          else:
              src = f"{WORK_DIR}/{pattern}"
              if os.path.exists(src):
                  shutil.copy2(src, BACKUP_DIR)
                  print(f"  Copied {pattern}")

      print(f"[{timestamp}] Backup complete!\n")

  # Start mdrun in background
  os.chdir(WORK_DIR)
  process = subprocess.Popen([
      "gmx", "mdrun", "-v", "-deffnm", "md",
      "-cpi", "md.cpt", "-ntomp", "12", "-nb", "gpu",
      "-nstlist", "1000", "-noappend"
  ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

  print("Simulation started! Auto-backup every 5 minutes.\n")

  # Monitor and backup loop
  backup_interval = 60  # 5 minutes in seconds
  last_backup = time.time()

  try:
      while process.poll() is None:  # While simulation is running
          # Print mdrun output
          line = process.stdout.readline()
          if line:
              print(line, end='')

          # Check if it's time to backup
          if time.time() - last_backup >= backup_interval:
              backup_to_drive()
              last_backup = time.time()

          time.sleep(0.1)

      # Final backup when simulation completes
      print("\n=== Simulation completed! Final backup... ===")
      backup_to_drive()
      print("All files saved to Google Drive!")

  except KeyboardInterrupt:
      print("\nInterrupted! Saving progress...")
      process.terminate()
      backup_to_drive()
      print("Progress saved to Google Drive!")

Streaming output truncated to the last 5000 lines.
step 4586500, will finish Fri Jan 30 05:59:07 2026
step 4586600, will finish Fri Jan 30 05:59:07 2026
step 4586700, will finish Fri Jan 30 05:59:07 2026
step 4586800, will finish Fri Jan 30 05:59:07 2026
step 4586900, will finish Fri Jan 30 05:59:07 2026
step 4587000, will finish Fri Jan 30 05:59:07 2026
step 4587100, will finish Fri Jan 30 05:59:07 2026
step 4587200, will finish Fri Jan 30 05:59:07 2026
step 4587300, will finish Fri Jan 30 05:59:07 2026
step 4587400, will finish Fri Jan 30 05:59:07 2026
step 4587500, will finish Fri Jan 30 05:59:07 2026
step 4587600, will finish Fri Jan 30 05:59:07 2026
step 4587700, will finish Fri Jan 30 05:59:07 2026
step 4587800, will finish Fri Jan 30 05:59:07 2026
step 4587900, will finish Fri Jan 30 05:59:07 2026
step 4588000, will finish Fri Jan 30 05:59:07 2026
step 4588100, will finish Fri Jan 30 05:59:07 2026
step 4588200, will finish Fri Jan 30 05:59:07 2026
step 4588300, will finish Fri J

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 7: Verify Results

**Time:** 10 seconds

Check simulation completed successfully.

In [9]:
import os

work_dir = '/content/md_simulation'

print("📊 Checking simulation results...\n")

# Check output files
output_files = ['md.xtc', 'md.log', 'md.edr', 'md.cpt']
for filename in output_files:
    filepath = f'{work_dir}/{filename}'
    if os.path.exists(filepath):
        size = os.path.getsize(filepath) / 1024 / 1024
        print(f"✅ {filename:12s} ({size:7.1f} MB)")
    else:
        print(f"❌ {filename:12s} MISSING")

# Check final step from log
print("\n📋 Final progress from log:")
!tail -100 {work_dir}/md.log | grep -E "Step|time" | tail -5

print("\n⚡ Performance achieved:")
!grep 'Performance' {work_dir}/md.log | tail -3

# Check for successful completion
print("\n📋 Completion status:")
!grep -i 'finished' {work_dir}/md.log || echo "⚠️  'Finished' not found - check manually"

print("\n" + "="*70)

📊 Checking simulation results...

❌ md.xtc       MISSING
❌ md.log       MISSING
❌ md.edr       MISSING
✅ md.cpt       (    8.5 MB)

📋 Final progress from log:
tail: cannot open '/content/md_simulation/md.log' for reading: No such file or directory

⚡ Performance achieved:
grep: /content/md_simulation/md.log: No such file or directory

📋 Completion status:
grep: /content/md_simulation/md.log: No such file or directory
⚠️  'Finished' not found - check manually



## Step 8: Save Results to Google Drive

**Time:** 5-10 minutes

**IMPORTANT:** Save before Colab disconnects!

In [10]:
%%time
import os
from datetime import datetime

work_dir = '/content/md_simulation'
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_dir = f'/content/drive/MyDrive/p53_md_results_{timestamp}'

print(f"📦 Saving results to Google Drive...\n")
print(f"   Destination: {results_dir}\n")

# Create results directory
!mkdir -p {results_dir}

# Copy all important files
files_to_save = ['md.xtc', 'md.log', 'md.edr', 'md.cpt', 'md.tpr', 'topol.top']
for filename in files_to_save:
    src = f'{work_dir}/{filename}'
    if os.path.exists(src):
        !cp {src} {results_dir}/
        size = os.path.getsize(src) / 1024 / 1024
        print(f"   ✅ Saved {filename:12s} ({size:7.1f} MB)")
    else:
        print(f"   ⚠️  {filename} not found")

# Create summary file
summary_path = f'{results_dir}/SUMMARY.txt'
with open(summary_path, 'w') as f:
    f.write("p53 MD Simulation Results\n")
    f.write("="*50 + "\n")
    f.write(f"Completed: {timestamp}\n")
    f.write(f"Platform: Google Colab (T4 GPU)\n")
    f.write(f"Simulation: A189S_M133L_S95T\n")
    f.write(f"\nFiles saved:\n")
    for filename in files_to_save:
        filepath = f'{work_dir}/{filename}'
        if os.path.exists(filepath):
            size = os.path.getsize(filepath) / 1024 / 1024
            f.write(f"  - {filename}: {size:.1f} MB\n")

print(f"\n✅ Results saved to Google Drive!")
print(f"\n📁 Location: {results_dir}")
print("\n📥 To download:")
print("   1. Go to: https://drive.google.com")
print(f"   2. Find folder: p53_md_results_{timestamp}")
print("   3. Right-click → Download (or select all files)")
print("\n" + "="*70)

📦 Saving results to Google Drive...

   Destination: /content/drive/MyDrive/p53_md_results_20260130_060052

   ⚠️  md.xtc not found
   ⚠️  md.log not found
   ⚠️  md.edr not found
   ✅ Saved md.cpt       (    8.5 MB)
   ✅ Saved md.tpr       (    9.5 MB)
   ⚠️  topol.top not found

✅ Results saved to Google Drive!

📁 Location: /content/drive/MyDrive/p53_md_results_20260130_060052

📥 To download:
   1. Go to: https://drive.google.com
   2. Find folder: p53_md_results_20260130_060052
   3. Right-click → Download (or select all files)

CPU times: user 9.59 ms, sys: 0 ns, total: 9.59 ms
Wall time: 311 ms


In [11]:
  from google.colab import drive
  import shutil
  import os

  # Mount Drive
  drive.mount('/content/drive')

  # Create output folder
  output_dir = "/content/drive/MyDrive/p53_md_final"
  os.makedirs(output_dir, exist_ok=True)

  # Files to copy
  files = [
      "md.tpr",
      "md.part0011.xtc",
      "md.part0012.xtc",
      "md.part0013.xtc"
  ]

  # Copy files
  for f in files:
      src = f"/content/md_simulation/{f}"
      if os.path.exists(src):
          print(f"Copying {f}...")
          shutil.copy2(src, output_dir)
          print(f"  ✓ Done ({os.path.getsize(src)/1e6:.1f} MB)")
      else:
          print(f"  ✗ {f} not found")

  print(f"\nAll files saved to: {output_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying md.tpr...
  ✓ Done (9.9 MB)
Copying md.part0011.xtc...
  ✓ Done (4.1 MB)
Copying md.part0012.xtc...
  ✓ Done (0.0 MB)
Copying md.part0013.xtc...
  ✓ Done (175.6 MB)

All files saved to: /content/drive/MyDrive/p53_md_final


In [ ]:
print("Attempting to kill any running 'gmx mdrun' processes...")
!pkill gmx_mpi # or gmx_d if using double precision
!pkill gmx # generic gmx process
print("Checked for and attempted to kill 'gmx mdrun' processes. If simulation was running in background, it should now be stopped.")


Attempting to kill any running 'gmx mdrun' processes...
Checked for and attempted to kill 'gmx mdrun' processes. If simulation was running in background, it should now be stopped.


---

## 🔄 Emergency: Save Checkpoint (If Timeout Approaching)

**Run this if:**
- Session will timeout (approaching 12 hours)
- Need to stop early
- Want to resume in new session

**How to use:**
1. Click **stop** (⏹️) on simulation cell
2. Wait 30 seconds
3. Run this cell

In [ ]:
import os
from datetime import datetime

work_dir = '/content/md_simulation'
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
checkpoint_dir = f'/content/drive/MyDrive/p53_checkpoint_{timestamp}'

print(f"💾 Saving checkpoint for resume...\n")

!mkdir -p {checkpoint_dir}

# Copy checkpoint files
!cp {work_dir}/md.cpt {checkpoint_dir}/ 2>/dev/null || echo "No checkpoint file"
!cp {work_dir}/md.tpr {checkpoint_dir}/
!cp {work_dir}/topol.top {checkpoint_dir}/
!cp {work_dir}/md.log {checkpoint_dir}/md_partial.log 2>/dev/null || echo "No log yet"
!cp {work_dir}/md.xtc {checkpoint_dir}/md_partial.xtc 2>/dev/null || echo "No trajectory yet"

# Show current progress
print("\n📊 Checkpoint saved at:")
!tail -10 {work_dir}/md.log 2>/dev/null | grep -E 'Step|time' | tail -2 || echo "Check log manually"

print(f"\n✅ Checkpoint saved to: {checkpoint_dir}")
print("\n🔄 To resume in new session:")
print("   1. Start new Colab session")
print("   2. Re-run Steps 1-4 (GPU check, compile, mount Drive)")
print(f"   3. Copy files from: {checkpoint_dir}")
print("   4. Run simulation with -cpi md.cpt flag")

---

## 🐛 Troubleshooting

### Error: "No GPU available"
```
Runtime → Change runtime type → GPU → Save
Then restart runtime and re-run all cells.
```

### Error: "gmx: command not found"
```
Re-run Step 3 (compile GROMACS).
Make sure compilation completed successfully.
```

### Error: "File not found: md.tpr"
```
1. Check files were uploaded correctly
2. Verify file sizes (md.tpr ~9 MB, not 1 KB)
3. If using Git LFS, files may be pointer files
```

### Error: "LINCS warning" or "NaN detected"
```
Checkpoint file may be corrupted.
Re-upload md.cpt from your Mac.
```

### Compilation fails at cmake
```
Run: !cat /tmp/cmake_output.log | tail -100
Look for error messages.
Common fix: Update cmake or check CUDA paths.
```

### Performance slower than expected
```
Expected: 150-250 ns/day on T4

Check GPU utilization:
!nvidia-smi

GPU-Util should be >90%.
If low, try reducing -ntomp threads.
```

### Session disconnects during simulation
```
1. Run checkpoint save cell above
2. Start new Colab session
3. Re-run Steps 1-5
4. Resume from checkpoint

Tip: Colab Pro ($10/month) has 24-hour sessions.
```

---

## 📊 Performance Comparison

| Platform | Hardware | Performance | 10 ns Time |
|----------|----------|-------------|------------|
| Mac | M1 Pro CPU | 26 ns/day | 12+ hours |
| **Colab** | **T4 GPU** | **150-250 ns/day** | **1-2 hours** |
| Windows | RTX 3060 | 150-250 ns/day | 1-2 hours |

**Speedup: 6-10× faster!** 🚀

---

## ✅ Success Checklist

**Before simulation:**
- [ ] GPU shows T4 (Step 1)
- [ ] GROMACS compiled with CUDA (Step 3b)
- [ ] Files uploaded and correct size (Step 5)

**During simulation:**
- [ ] Progress updates showing
- [ ] Performance 150-250 ns/day
- [ ] No errors in output

**After simulation:**
- [ ] md.xtc is 500-1000 MB
- [ ] Log shows "Finished mdrun"
- [ ] Results saved to Drive

---

## 📚 Sources

- [GROMACS Installation Guide](https://manual.gromacs.org/current/install-guide/index.html)
- [GROMACS Conda GPU Issue](https://gromacs.bioexcel.eu/t/gromacs-install-with-conda-but-cant-run-on-gpu/10856)
- [bioinfkaustin/gromacs-on-colab](https://github.com/bioinfkaustin/gromacs-on-colab)
- [TheBiomics GROMACS GPU Guide](https://www.thebiomics.com/research/gromacs-installation-with-gpu.html)

---

*Created: 2026-01-27*  
*Project: p53 StabiliMut Initiative 2*  
*GROMACS: 2024.4 with CUDA*  
*Platform: Google Colab (Free T4 GPU)*
